In [1]:
import pandas as pd
import re

# Load dataset
df = pd.read_csv("Blood Donation_questions_and_answers.csv")

# Keep only the needed columns
df = df[["Question", "Answer"]].copy()

# Remove prefixes like "Question 1:"
df["Question"] = df["Question"].astype(str).str.replace(
    r"^Question\s*\d+\s*:\s*", "", regex=True
)

# Normalize spacing
df["Question"] = df["Question"].str.strip().str.replace(r"\s+", " ", regex=True)
df["Answer"] = df["Answer"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Check whether same question has multiple different answers
conflicts = (
    df.groupby("Question")["Answer"]
    .nunique()
    .reset_index(name="answer_count")
)

conflicting_questions = conflicts[conflicts["answer_count"] > 1]

print("Cleaned dataset shape:", df.shape)
print("Questions with multiple different answers:", len(conflicting_questions))

# Save cleaned dataset
df.to_csv("blood_donation_qa_cleaned.csv", index=False)
print("Saved as blood_donation_qa_cleaned.csv")

Cleaned dataset shape: (3416, 2)
Questions with multiple different answers: 36
Saved as blood_donation_qa_cleaned.csv


Resolve conflicting answers for identical questions

In [2]:
# Show conflicting questions and their answers

conflicting_qas = df[df["Question"].isin(conflicting_questions["Question"])]

conflicting_qas = conflicting_qas.sort_values("Question")

conflicting_qas.to_csv("conflicting_questions_review.csv", index=False)

print("Saved conflicting questions to conflicting_questions_review.csv")

Saved conflicting questions to conflicting_questions_review.csv


Automatic conflict resolution strategy

In [3]:
import pandas as pd

df = pd.read_csv("blood_donation_qa_cleaned.csv")

# Identify questions with multiple answers
conflict_counts = df.groupby("Question")["Answer"].nunique()

conflict_questions = conflict_counts[conflict_counts > 1].index

# Resolve conflicts by keeping the longest answer
resolved_df = (
    df.sort_values("Answer", key=lambda x: x.str.len(), ascending=False)
      .drop_duplicates(subset="Question", keep="first")
      .reset_index(drop=True)
)

print("Final dataset shape:", resolved_df.shape)

resolved_df.to_csv("blood_donation_qa_final.csv", index=False)

print("Saved cleaned dataset as blood_donation_qa_final.csv")

Final dataset shape: (3376, 2)
Saved cleaned dataset as blood_donation_qa_final.csv


Creating training, validation, and test splits

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load final dataset
df = pd.read_csv("blood_donation_qa_final.csv")

# First split: train (80%) vs temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: validation (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

# Save splits
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)
print("Test size:", test_df.shape)

Train size: (2700, 2)
Validation size: (338, 2)
Test size: (338, 2)
